**平移不变性**：分类器不应该对不同像素位置的相同物体计算结果不同  
**局部性**：分类器不需要关心其他位置的像素信息，只需要关心局部信息，远位置信息与该位置几乎无关

h_i,j:输出
x_k,l:输入，k,l：坐标

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

核函数的偏移和大小是超参数

In [2]:
def corr2d(X,k):
    h,w = k.shape
    Y = torch.zeros(X.shape[0]-h+1,X.shape[1]-w+1)
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j] = (X[i:i+h,j:j+w]*k).sum()
    return Y 

In [3]:
X = torch.arange(9).reshape(3,3)
Y = torch.arange(4).reshape(2,2)
X,Y,corr2d(X,Y)

(tensor([[0, 1, 2],
         [3, 4, 5],
         [6, 7, 8]]),
 tensor([[0, 1],
         [2, 3]]),
 tensor([[19., 25.],
         [37., 43.]]))

In [4]:
class Conv2D(nn.Module):
    def __init__(self,Kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(Kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))
    def forward(self,X):
        return corr2d(X,self.weight)+self.bias

In [5]:
X = torch.ones(6,8)
X[:,2:6] = 0
K = torch.tensor([[1.0,-1.0]])
Y = corr2d(X,K)
X,Y

(tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
         [1., 1., 0., 0., 0., 0., 1., 1.],
         [1., 1., 0., 0., 0., 0., 1., 1.],
         [1., 1., 0., 0., 0., 0., 1., 1.],
         [1., 1., 0., 0., 0., 0., 1., 1.],
         [1., 1., 0., 0., 0., 0., 1., 1.]]),
 tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
         [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
         [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
         [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
         [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
         [ 0.,  1.,  0.,  0.,  0., -1.,  0.]]))

In [6]:
conv2d = nn.Conv2d(1,1,kernel_size=(1,2),bias=False)
# 第一个是通道维，第二个是批量大小维
X = X.reshape(1,1,6,8)
Y = Y.reshape(1,1,6,7)

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y-Y_hat)**2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= 3e-2* conv2d.weight.grad
    print(f'epoch:{i+1},loss:{l.sum():.3f}')
    

epoch:1,loss:18.580
epoch:2,loss:7.673
epoch:3,loss:3.183
epoch:4,loss:1.330
epoch:5,loss:0.561
epoch:6,loss:0.240
epoch:7,loss:0.105
epoch:8,loss:0.047
epoch:9,loss:0.022
epoch:10,loss:0.011


In [7]:
conv2d.weight.data

tensor([[[[ 0.9785, -0.9930]]]])

In [8]:
D = torch.ones(1,3,6,8)
D[:,2,:,:] = 0
D.shape

torch.Size([1, 3, 6, 8])

In [9]:
condv2d = nn.Conv2d(3,3,kernel_size=3,padding=1,stride=2,bias=False)
condv2d.weight.data,condv2d.weight.data.shape

(tensor([[[[ 0.0229, -0.0285, -0.0615],
           [-0.0928, -0.1480,  0.1304],
           [ 0.1404, -0.0412, -0.1889]],
 
          [[-0.1909, -0.1361, -0.0279],
           [ 0.0744,  0.0327,  0.0370],
           [ 0.0869, -0.0206, -0.1366]],
 
          [[-0.1793, -0.1864, -0.1437],
           [ 0.1131,  0.0635,  0.1882],
           [-0.1776,  0.1213, -0.0355]]],
 
 
         [[[ 0.0788,  0.1582,  0.1693],
           [ 0.1679, -0.1688,  0.0253],
           [ 0.1876, -0.1835, -0.0538]],
 
          [[-0.1347,  0.1713, -0.0956],
           [-0.0210, -0.0843, -0.0289],
           [ 0.1800,  0.0356,  0.1847]],
 
          [[-0.0331,  0.1091,  0.1245],
           [-0.0486,  0.0974, -0.1726],
           [-0.0105,  0.0562,  0.1188]]],
 
 
         [[[-0.0253, -0.1304,  0.1325],
           [ 0.1715,  0.1690, -0.1810],
           [-0.0651,  0.0949,  0.0829]],
 
          [[ 0.1714,  0.0636,  0.0710],
           [ 0.1759,  0.0358, -0.1544],
           [-0.1359,  0.0476, -0.1231]],
 
          

In [10]:
condv2d(D)

tensor([[[[-0.3352, -0.1262, -0.1262, -0.1262],
          [-0.5892, -0.5484, -0.5484, -0.5484],
          [-0.5892, -0.5484, -0.5484, -0.5484]],

         [[-0.2738,  0.2406,  0.2406,  0.2406],
          [ 0.1294,  0.5880,  0.5880,  0.5880],
          [ 0.1294,  0.5880,  0.5880,  0.5880]],

         [[-0.0281,  0.1183,  0.1183,  0.1183],
          [ 0.1087,  0.4012,  0.4012,  0.4012],
          [ 0.1087,  0.4012,  0.4012,  0.4012]]]],
       grad_fn=<ConvolutionBackward0>)

In [11]:
# layer_data0 = condv2d.weight.data[0,...]
# layer_data1 = condv2d.weight.data[1,...]
# layer_data2 = condv2d.weight.data[2,...]
# layer_data0.shape

In [12]:
layer0 = nn.Conv2d(3,1,kernel_size=3,padding=1,stride=2,bias=False)
layer1 = nn.Conv2d(3,1,kernel_size=3,padding=1,stride=2,bias=False)
layer2 = nn.Conv2d(3,1,kernel_size=3,padding=1,stride=2,bias=False)

layer0.weight.data[0,...]= condv2d.weight.data[0,...]
layer1.weight.data[0,...] = condv2d.weight.data[1,...]
layer2.weight.data[0,...] = condv2d.weight.data[2,...]

layer0.weight.data,layer0.weight.data.shape

(tensor([[[[ 0.0229, -0.0285, -0.0615],
           [-0.0928, -0.1480,  0.1304],
           [ 0.1404, -0.0412, -0.1889]],
 
          [[-0.1909, -0.1361, -0.0279],
           [ 0.0744,  0.0327,  0.0370],
           [ 0.0869, -0.0206, -0.1366]],
 
          [[-0.1793, -0.1864, -0.1437],
           [ 0.1131,  0.0635,  0.1882],
           [-0.1776,  0.1213, -0.0355]]]]),
 torch.Size([1, 3, 3, 3]))

In [34]:
layer0(D),torch.allclose(layer0(D)[:,0,...],condv2d(D)[:,0,...])

(tensor([[[[-0.3352, -0.1262, -0.1262, -0.1262],
           [-0.5892, -0.5484, -0.5484, -0.5484],
           [-0.5892, -0.5484, -0.5484, -0.5484]]]],
        grad_fn=<ConvolutionBackward0>),
 True)